In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from phase import PhaseSolver, PhaseConfig



In [ ]:
from dataclasses import dataclass

@dataclass
class SimConfig:
    N: int
    H: int
    W: int
    alpha_frame_std: float
    noise_std: float
    gamma_frame_base: float
    gamma_frame_std: float
    gamma_radius: float
    delta: str | np.ndarray
    kx: float
    ky: float
    kxx: float
    kyy: float
    kxy: float
    rng: np.random.Generator | None = None
    
    def __post_init__(self):
        self.rng = self.rng if self.rng is not None else np.random.default_rng()
        self.X, self.Y = self.grid()

        if isinstance(self.delta, str):
            mode = self.delta.lower()
            if mode == "uniform":
                self.delta = np.linspace(0, 2 * np.pi, self.N, endpoint=False)
            elif mode == "random":
                self.delta = self.rng.uniform(0, 2 * np.pi, self.N)
                self.delta[0] = 0.0
            else:
                raise ValueError(f"Unknown delta mode: {self.delta!r}")
        elif isinstance(self.delta, np.ndarray):
            if self.delta.shape != (self.N,):
                raise ValueError(f"delta shape must be ({self.N},), got {self.delta.shape}")
        else:
            raise ValueError(f"delta must be str or np.ndarray, got {type(self.delta)}")
    
    def grid(self):
        Y, X = np.mgrid[0:self.H, 0:self.W].astype(float)
        return X, Y
    
    def gamma(self):
        r2 = (self.X - self.W/2)**2 + (self.Y - self.H/2)**2
        gamma_spatial = np.exp(-r2 / (2 * self.gamma_radius**2))  # (H, W)
        g = self.gamma_frame_base + self.gamma_frame_std * self.rng.standard_normal(self.N)
        return g[:, np.newaxis, np.newaxis] * gamma_spatial[np.newaxis, :, :]
    
    def alpha(self):
        return 1.0 + self.alpha_frame_std * self.rng.standard_normal(self.N)
    
    def _check_I(self, I1: np.ndarray, I2: np.ndarray):
        if I1.shape != I2.shape:
            raise ValueError(f"I1 and I2 shapes must be equal, got {I1.shape} and {I2.shape}")
        if I1.shape != (self.H, self.W):
            raise ValueError(f"I1 and I2 shapes must be ({self.H}, {self.W}), got {I1.shape}")
    
    def a(self, I1: np.ndarray, I2: np.ndarray):
        self._check_I(I1, I2)
        return I1 + I2
    
    def b(self, I1: np.ndarray, I2: np.ndarray):
        self._check_I(I1, I2)
        return 2 * self.gamma() * np.sqrt(I1[np.newaxis, :, :] * I2[np.newaxis, :, :])
    
    def carrier(self):
        return self.kx * self.X + self.ky * self.Y + self.kxx * self.X**2 + self.kyy * self.Y**2 + self.kxy * self.X * self.Y
    
    def phase(self, phi: np.ndarray):
        if phi.shape != (self.H, self.W):
            raise ValueError(f"phi shape must be ({self.H}, {self.W}), got {phi.shape}")
        return phi[np.newaxis, :, :] + self.carrier()[np.newaxis, :, :] + self.delta[:, np.newaxis, np.newaxis]
    
    def I_t(self, I1: np.ndarray, I2: np.ndarray, phi: np.ndarray):
        base = self.a(I1, I2)[np.newaxis, :, :] + self.b(I1, I2) * np.cos(self.phase(phi))
        return self.alpha()[:, np.newaxis, np.newaxis] * base
    
    def I(self, I1: np.ndarray, I2: np.ndarray, phi: np.ndarray):
        return self.I_t(I1, I2, phi) + self.noise_std * self.rng.standard_normal((self.N, self.H, self.W))
        

In [ ]:
# Define params
N = 5
H = 512
W = 512
alpha_frame_std = 0.0
noise_std = 0.0
gamma_frame_base = 0.5
gamma_frame_std = 0.0
gamma_radius = H / 3.0
delta = "uniform"
kx = 2 * np.pi * 10 / W
ky = 2 * np.pi * 10 / H
kxx = 0.0
kyy = 0.0
kxy = 0.0

# Create config
sim = SimConfig(N, H, W, alpha_frame_std, noise_std, gamma_frame_base, gamma_frame_std, 
                gamma_radius, delta, kx, ky, kxx, kyy, kxy)

# Define Intensity and phase
I1_base = 120.0 * np.ones((H, W))
I2 = 80.0 * np.ones((H, W))

# --- Square test object (known transmission and phase), placed in the I1 beam ---
X, Y = sim.grid()
obj_half_size = H // 16
obj_cy, obj_cx = H // 2, W // 2
square_mask = ((np.abs(Y - obj_cy) <= obj_half_size) &
            (np.abs(X - obj_cx) <= obj_half_size))
object_transmission = 0.6  # fraction of I1 transmitted through the object
object_phase = 1.2         # known phase shift introduced by the object, in radians

I1 = np.where(square_mask, I1_base * object_transmission, I1_base)
phi = np.where(square_mask, object_phase, 0.0)

stack = sim.I_t(I1, I2, phi)

num = 0
plt.subplot(121)
plt.imshow(stack[num])

plt.subplot(122)
plt.plot(stack[num, 256])

In [ ]:
stack.shape

In [ ]:
def build_model(N, H, W, gamma_frame_std=0.0, alpha_frame_std=0.0, rng=None):
    rng = rng if rng is not None else np.random.default_rng()
    
    # Ground truth
    real_delta = np.linspace(0, 2*np.pi, N, endpoint=False)

    Y, X = np.mgrid[0:H, 0:W].astype(float)
    r2 = (X - W/2)**2 + (Y - H/2)**2

    # --- alpha_n: per-frame source-power factor, shifts from 1 with std
    # alpha_frame_std -- returned separately and passed to simulate_stack's
    # own `alpha=` parameter (it already exists for exactly this), not baked
    # into I1/I2. Baking it into I1/I2 made them (and hence a, b below)
    # per-frame (N,H,W) arrays instead of the static (H,W) maps simulate_stack
    # expects -- that's what crashed (`too many values to unpack` in its own
    # `N, (H, W) = len(delta), a.shape`).
    real_alpha = 1.0 + alpha_frame_std * rng.standard_normal(N)  # (N,)

    # --- Beam intensities (I1 = sample-arm beam, I2 = reference-arm beam) ---
    I1_base = 120.0 * np.ones((H, W))
    I2 = 80.0 * np.ones((H, W))

    # --- Square test object (known transmission and phase), placed in the I1 beam ---
    obj_half_size = H // 8
    obj_cy, obj_cx = H // 2, W // 2
    square_mask = ((np.abs(Y - obj_cy) <= obj_half_size) &
                (np.abs(X - obj_cx) <= obj_half_size))
    object_transmission = 0.6  # fraction of I1 transmitted through the object
    object_phase = 1.2         # known phase shift introduced by the object, in radians

    I1 = np.where(square_mask, I1_base * object_transmission, I1_base)

    # --- Coherence: a fixed spatial falloff from the field center, times an
    # independent per-frame random scale -- shifts from 1 with std
    # `gamma_frame_std`, e.g. simulating frame-to-frame source-coherence
    # jitter on top of the fixed spatial structure. gamma_frame_std=0 (default)
    # reproduces the old purely-spatial, frame-independent behavior. Combined
    # shape (N, H, W); simulate_stack broadcasts it correctly either way.
    radius = H / 1.2
    gamma_spatial = np.exp(-r2 / (2 * radius**2))  # (H, W)
    real_g = 1.0 + gamma_frame_std * rng.standard_normal(N)  # (N,)
    real_gamma = real_g[:, None, None] * gamma_spatial[None, :, :]  # (N, H, W)

    # --- a, b derived from the two beam intensities (Eq. 1 of docs/interference_model.md:
    # a is "the two beam intensities", b is "the beam intensities and the coherence
    # envelope"). b is the *nominal* fringe amplitude (coherence = 1) -- the full
    # coherence envelope (spatial x per-frame) is applied separately via `gamma` in
    # simulate_stack below, not baked in here too. (Previously it was baked into
    # both b and passed again as gamma, which double-counted it -- effectively
    # squaring gamma in the simulated stack. Fixed here.) ---
    real_a = I1 + I2
    real_b = 2 * np.sqrt(I1 * I2)

    # --- Carrier phase: linear tilt (kx, ky -> straight/line fringes) plus quadratic
    # curvature (kxx, kyy, kxy -> elliptical fringes; kxx==kyy and kxy==0 gives
    # circular fringes), matching phase/carrier.py's CarrierResult parameterization.
    # Keep the tilt frequency comfortably above dc_radius (default 8 bins) so
    # measure_frame_contrast's carrier-peak detection finds a genuine peak.
    carrier_cycles_x, carrier_cycles_y = -20, -20
    kx, ky = 2*np.pi*carrier_cycles_x/W, 2*np.pi*carrier_cycles_y/H
    kxx, kyy, kxy = 1e-4, 1e-4, 1e-5
    carrier = kx*X + ky*Y + kxx*X**2 + kyy*Y**2 + kxy*X*Y

    # --- Full phase map: carrier + the object's known phase shift, present only
    # inside the square ---
    real_phi = carrier + np.where(square_mask, object_phase, 0.0)
    
    return real_a, real_b, real_phi, real_delta, real_gamma, real_alpha

In [ ]:
rng = np.random.default_rng(0)
noise_std = 5.0
real_a, real_b, real_phi, real_delta, real_gamma, real_alpha = build_model(
    N=10, H=1024, W=1024, gamma_frame_std=0.1, alpha_frame_std=0.1, rng=rng)
stack = simulate_stack(real_a, real_b, real_phi, real_delta,
                        gamma=real_gamma, alpha=real_alpha, noise_std=noise_std, rng=rng)

plt.imshow(stack[1])
plt.title("Synthetic frame 0")
plt.colorbar()

In [ ]:
# Init solver
cfg = PhaseConfig().from_yaml("phase_config.yaml")
cfg.use_alpha = True
cfg.use_g = True
cfg.dc_radius = 10
solver = PhaseSolver(cfg)

# Solve
result = solver.fit(stack).result_
result.print_summary()

In [ ]:
# Test how changes with N:
reconstruction_error = []
N_range = [3, 5, 7, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80]
for i in N_range:
    real_a, real_b, real_phi, real_delta, real_gamma, real_alpha = build_model(
        N=i, H=1024, W=1024, gamma_frame_std=0.1, alpha_frame_std=0.1, rng=rng)
    stack = simulate_stack(real_a, real_b, real_phi, real_delta,
                            gamma=real_gamma, alpha=real_alpha, noise_std=noise_std, rng=rng)
    result = solver.fit(stack).result_
    reconstruction_error.append(result.reconstruction_error)
    
reconstruction_error = np.asarray(reconstruction_error)

plt.plot(N_range, reconstruction_error)